# H-mode global confinement $\tau_E$ scaling
Reproduces the basic engineering-variable results of Verdoolaege *et al.* 2021, Nucl. Fusion **61** 076006 (DB5.2.3) from the IMAS-migrated H-mode database.

In [52]:
import numpy as np

import _simdb_common as sc

db = sc.get_db()
sims = sc.query_dataset(db, "hmode")
print(f"{len(sims)} shots in hmode")

6394 shots in hmode


## Scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| $W_{th}$ | `summary/global_quantities/energy_thermal/value` | J |
| $\tau_{E,th}$ (TAUTH) | `summary/global_quantities/tau_energy/value` | s |
| $I_p$ | `summary/global_quantities/ip/value` | A |
| $B_t$ | `summary/global_quantities/b0/value` | T |
| $\bar n_e$ | `summary/line_average/n_e/value` | m^-3 |
| $P_{l,th}$ | `summary/global_quantities/power_loss/value` | W |
| $R_{geo}$ | `summary/boundary/geometric_axis_r/value` | m |
| $a$ | `summary/boundary/minor_radius/value` | m |
| $S$ | `temporary` (`AREA`) | m^2 |
| $V$ | `summary/global_quantities/volume/value` | m^3 |
| $M_{eff}$ | `summary/volume_average/meff_hydrogenic/value` | AMU |
| $\delta$ | `temporary` (`DELTA`) | — |
| TAUC93 | `temporary` (`TAUC93`) | — |
| $q_{95}$ | `summary/global_quantities/q_95/value` | — |
| $\kappa$ | `summary/boundary/elongation/value` | — |
| $Z_{eff}$ | `summary/line_average/zeff/value` | — |
| PGASZ | `summary/volume_average/zeff/value` | — |
| TOK | `summary/machine` | — |
| PHASE | `temporary/constant_string0d` (by `identifier.name`) | — |
| SELDB5 | `temporary/constant_integer0d` (by `identifier.name`) | — |

Derived (paper definitions): $\kappa_a = S/(\pi a^2)$, $\epsilon = a/R_{geo}$.


In [53]:
rows = []
for i, sim in enumerate(sims, start=1):
    if i % 100 == 0 or i == len(sims):
        print(f"\r  {i}/{len(sims)} shots ({sim.alias})".ljust(80), end="", flush=True)

    md = sc.load(sim)
    machine = md.get("machine", "")
    seldb5 = sc.temp(md, "SELDB5", n=0)
    n = len(seldb5)
    if n == 0:
        continue
    wth    = sc.path(md, "global_quantities", "energy_thermal", "value", n=n)
    tau    = sc.path(md, "global_quantities", "tau_energy", "value", n=n)
    tauth2 = sc.temp(md, "TAUTH2", n=n)  # fallback for tau when TAUTH is missing (see logbook section 8)
    tauth1 = sc.temp(md, "TAUTH1", n=n)  # second fallback
    ip    = sc.path(md, "global_quantities", "ip", "value", n=n)
    bt    = np.abs(sc.path(md, "global_quantities", "b0", "value", n=n))
    nel   = sc.path(md, "line_average", "n_e", "value", n=n)
    plth  = sc.path(md, "global_quantities", "power_loss", "value", n=n)
    rgeo  = sc.path(md, "boundary", "geometric_axis_r", "value", n=n)
    vol   = sc.path(md, "global_quantities", "volume", "value", n=n)
    amin  = sc.path(md, "boundary", "minor_radius", "value", n=n)
    area  = sc.temp(md, "AREA", n=n)
    meff  = sc.path(md, "volume_average", "meff_hydrogenic", "value", n=n)
    delta = sc.temp(md, "DELTA", n=n)
    tauc93 = sc.temp(md, "TAUC93", n=n)   # closed-divertor correction (ASDEX, PDX)
    q95   = sc.path(md, "global_quantities", "q_95", "value", n=n)
    kappa = sc.path(md, "boundary", "elongation", "value", n=n)
    zeff  = sc.path(md, "line_average", "zeff", "value", n=n)
    pgasz = sc.path(md, "volume_average", "zeff", "value", n=n)   # charge of the plasma working gas
    phase_raw = sc.temp_str(md, "PHASE", n=n)
    limmat = sc.path(md, "limiter", "material", "index", n=n)      # AUG/AUG-W split (limiter material index; W=2)
    if limmat.ndim == 0:                                           # shot-constant scalar -> broadcast to timeslices
        limmat = np.full(n, float(limmat))
    divmat = sc.temp_str(md, "DIVMAT", n=n)                        # JET-C/JET-ILW split (divertor material; "W"=ILW)
    if divmat.ndim == 0:                                           # shot-constant scalar -> broadcast to timeslices
        divmat = np.full(n, divmat.item(), dtype=object)

    n_slices = min(len(a) for a in (seldb5, wth, tau, tauth2, tauth1, ip, bt, nel, plth, rgeo, vol, amin, area,
                                     meff, delta, tauc93, q95, kappa, zeff, pgasz, phase_raw, limmat, divmat))
    if n_slices == 0:
        continue
    for j in range(n_slices):
        rows.append((machine, wth[j], tau[j], tauth2[j], tauth1[j], ip[j], bt[j], nel[j], plth[j], rgeo[j], vol[j],
                     amin[j], area[j], meff[j], delta[j], tauc93[j], q95[j], kappa[j], zeff[j], pgasz[j], phase_raw[j], seldb5[j],
                     limmat[j], divmat[j]))
print()

dtype = [("tok", "U16"), ("wth", float), ("tau", float), ("tauth2", float), ("tauth1", float), ("ip", float), ("bt", float),
         ("nel", float), ("plth", float), ("rgeo", float), ("vol", float), ("amin", float), ("area", float),
         ("meff", float), ("delta", float), ("tauc93", float), ("q95", float), ("kappa", float), ("zeff", float), ("pgasz", float),
         ("phase", object), ("seldb5", float), ("limmat", float), ("divmat", object)]
data = np.array(rows, dtype=dtype)

N      = len(data)
WTH    = data["wth"]
TAU    = data["tau"]
TAUTH2 = data["tauth2"]
TAUTH1 = data["tauth1"]
IP     = data["ip"]
BT     = data["bt"]
NEL    = data["nel"]
PLTH   = data["plth"]
RGEO   = data["rgeo"]
VOL    = data["vol"]
AMIN   = data["amin"]
AREA   = data["area"]
MEFF   = data["meff"]
DELTA  = data["delta"]
TAUC93 = data["tauc93"]
Q95    = data["q95"]
KAPPA  = data["kappa"]
ZEFF   = data["zeff"]
PGASZ  = data["pgasz"]
TOK    = data["tok"]
PHASE  = data["phase"]
SELDB5 = data["seldb5"]
LIMMAT = data["limmat"]
DIVMAT = data["divmat"]

print("Done.")

  6394/6394 shots (hmode/TUMAN3M/2000032308)                                   
Done.


In [ ]:
# Units and derived variables (formula in paper)
# tau_s = W_th / P_l,th, falling back to TAUTH then TAUTH2/TAUTH1 where W_th is missing.
wth_ok    = np.isfinite(WTH) & (WTH > 0) & np.isfinite(PLTH) & (PLTH > 0)
tau_ok    = np.isfinite(TAU) & (TAU > 0)
tauth2_ok = np.isfinite(TAUTH2) & (TAUTH2 > 0)
tauth1_ok = np.isfinite(TAUTH1) & (TAUTH1 > 0)

with np.errstate(invalid="ignore", divide="ignore"):
    tau_s = np.where(wth_ok, WTH / PLTH,
             np.where(tau_ok, TAU, np.where(tauth2_ok, TAUTH2, np.where(tauth1_ok, TAUTH1, np.nan))))
print(f"tau_s: {np.sum(wth_ok)} from W_th/P_l,th, {np.sum(~wth_ok & tau_ok)} from TAUTH, "
      f"{np.sum(~wth_ok & ~tau_ok & tauth2_ok)} from TAUTH2, {np.sum(~wth_ok & ~tau_ok & ~tauth2_ok & tauth1_ok)} from TAUTH1")

tauc93 = np.where(np.isfinite(TAUC93), TAUC93, 1.0)   # closed-divertor correction, paper section 2.2.1
tau_s = tau_s * tauc93
print(f"TAUC93 applied to {np.sum(tauc93 != 1.0)} timeslices "
      f"({', '.join(sorted(set(TOK[tauc93 != 1.0])))})")

ip_ma    = np.abs(IP) / 1e6                     # [MA]
Bt_T     = np.abs(BT)                           # [T]
ne_19    = NEL / 1e19                           # [10^19 m^-3]
Ploss_MW = PLTH / 1e6                           # [MW]
area_ok  = np.isfinite(AREA) & (AREA > 0)
kappa_a  = np.where(area_ok, AREA / (np.pi * AMIN**2),
                    VOL / (2.0 * np.pi * RGEO * np.pi * AMIN**2))    # volume form where AREA is absent
eps      = AMIN / RGEO                          # inverse aspect ratio
one_delta = 1.0 + DELTA                         # 1 + delta

# Subset selection: STD5 standard set, ELMy H-mode
PHASE_str = PHASE.astype(str)
std5 = (SELDB5 == 1)
elmy = np.char.startswith(PHASE_str, "HG") | np.char.startswith(PHASE_str, "HS")

helium = (PGASZ == 2)   # left out of the paper's analysis (section 2.2.1)

sel = std5 & elmy & ~helium

# ITER-like subset (paper section 2.1); a missing q95 or Zeff does not reject a slice
iter_like = (sel & np.isfinite(one_delta) & (one_delta > 0)
             & ~(Q95 <= 2.8) & (KAPPA > 1.3) & (KAPPA < 2.2) & (eps < 0.5) & ~(ZEFF >= 5))

print(f"Total pulses       : {N}")
print(f"SELDB5 == 1        : {np.sum(std5)}")
print(f"ELMy H (HG/HS)     : {np.sum(elmy)}")
print(f"helium excluded    : {np.sum(helium & std5 & elmy)}")
print(f"std5 + ELMy + valid: {np.sum(sel)}   (paper 6233)")
print(f"  of which ITER-like: {np.sum(iter_like)}   (paper 5260)")

# Wall/divertor-material splits (paper Table 1): AUG-W by limiter material index (W=2),
# JET-ILW by divertor material string ("W").
DIVMAT_str = DIVMAT.astype(str)
is_aug_w   = (TOK == "AUG") & (LIMMAT == 2)
is_jet_ilw = (TOK == "JET") & (DIVMAT_str == "W")

print("\nper machine (ELMy valid / ITER-like):")
for tok in np.unique(TOK[sel]):
    print(f"  {tok:10s}: {np.sum((TOK == tok) & sel):5d} {np.sum((TOK == tok) & iter_like):5d}")
    if tok == "AUG":
        print(f"    {'AUG (C/C-W)':10s}: {np.sum((TOK == tok) & ~is_aug_w & sel):5d} {np.sum((TOK == tok) & ~is_aug_w & iter_like):5d}   (paper 1377 / 1370)")
        print(f"    {'AUG-W':10s}: {np.sum(is_aug_w & sel):5d} {np.sum(is_aug_w & iter_like):5d}   (paper 767 / 767)")
    if tok == "JET":
        print(f"    {'JET-C':10s}: {np.sum((TOK == tok) & ~is_jet_ilw & sel):5d} {np.sum((TOK == tok) & ~is_jet_ilw & iter_like):5d}   (paper 1762 / 1606)")
        print(f"    {'JET-ILW':10s}: {np.sum(is_jet_ilw & sel):5d} {np.sum(is_jet_ilw & iter_like):5d}   (paper 866 / 855)")


tau_s: 13830 from W_th/P_l,th, 0 from TAUTH, 46 from TAUTH2, 0 from TAUTH1
TAUC93 applied to 742 timeslices (ASDEX, PDX)
Total pulses       : 14153
SELDB5 == 1        : 7568
ELMy H (HG/HS)     : 7520
helium excluded    : 36
std5 + ELMy + valid: 6228   (paper 6233)
  of which ITER-like: 5249   (paper 5260)

per machine (ELMy valid / ITER-like):
  ASDEX     :   431     0
  AUG       :  2137  2130
    AUG (C/C-W):  1377  1370   (paper 1377 / 1370)
    AUG-W     :   760   760   (paper 767 / 767)
  CMOD      :    45    45
  COMPASS   :    16    15
  D3D       :   388   383
  JET       :  2626  2458
    JET-C     :  1760  1603   (paper 1762 / 1606)
    JET-ILW   :   866   855   (paper 866 / 855)
  JFT2M     :    70    59
  JT60U     :   100   100
  MAST      :    43     0
  NSTX      :   185     0
  PBXM      :    59    59
  PDX       :    97     0
  START     :     8     0
  TCV       :    11     0
  TDEV      :    10     0
  TFTR      :     2     0


The STD5+ELMy count (6228) is 5 short of the paper's 6233. 

This is not a "valid data" filtering effect. `std5 & elmy & ~helium` alone already gives 6228 with nothing else applied; dropping the helium exclusion would instead overshoot the count to 6264.
The residual -5 can be identified per-machine from paper Table 1's STD5-ELMy-H column:

| machine | this notebook | paper | difference |
|---|---|---|---|
| AUG (C/C-W) | 1377 | 1377 | 0 |
| AUG-W | 760 | 767 | **-7** |
| JET-C | 1760 | 1762 | -2 |
| JET-ILW | 866 | 866 | 0 |
| JFT-2M | 70 | 69 | +1 |
| TdeV | 10 | 7 | +3 |

-7-2+1+3 = -5 (the total gap). 
The AUG-W difference is the dominant term, with 7 timeslices having `PGASZ=2` with `PGASA=4` (confirmed against the raw CSV, not mapping artifact), so our (paper-consistent) helium exclusion drops them. 

The paper's own Table 8 lists AUG-W's `n_obs` as exactly 767, i.e. including these 7, despite section 2.2.1's statement that helium plasmas are left out "in the present analysis of the database and all confinement scalings." This could be an undocumented exception in the paper's stated methodology. JET-C's -2 may be a smaller instance of the same thing. JFT-2M (+1) and TdeV (+3) are unexplained residuals of the same size as the raw `SELDB5` flag mismatches noted earlier, not helium-related.


## Table 2 — engineering-variable ranges (STD5 ELMy H)
Compare with paper Table 2 (DB5.2.3-STD5 ELMy H).


In [66]:
cols = {
    "tau_E,th [s]": tau_s[sel],
    "Ip [MA]":      ip_ma[sel],
    "Bt [T]":       Bt_T[sel],
    "ne [1e19]":    ne_19[sel],
    "Pl,th [MW]":   Ploss_MW[sel],
    "Rgeo [m]":     RGEO[sel],
    "1+delta":      one_delta[sel],
    "kappa_a":      kappa_a[sel],
    "eps":          eps[sel],
    "Meff":         MEFF[sel],
}
print(f"{'variable':14s} {'min':>9s} {'max':>9s} {'mean':>9s} {'median':>9s} {'std':>9s} {'n':>6s}")
for name, x in cols.items():
    n_nan = np.sum(~np.isfinite(x))
    print(f"{name:14s} {np.nanmin(x):9.4g} {np.nanmax(x):9.4g} {np.nanmean(x):9.4g} {np.nanmedian(x):9.4g} "
          f"{np.nanstd(x):9.4g} {len(x)-n_nan:>6d}" + (f"  ({n_nan} NaN)" if n_nan else ""))

variable             min       max      mean    median       std      n
tau_E,th [s]    0.002237     1.321    0.1752    0.1223    0.1491   6228
Ip [MA]           0.1593     5.134     1.386     1.007    0.8111   6228
Bt [T]            0.2613     5.821     2.143     2.191    0.6646   6228
ne [1e19]          1.166      42.6     5.949     5.502       3.2   6228
Pl,th [MW]        0.1464     32.67     7.991     6.598     5.354   6228
Rgeo [m]          0.2804       3.4     2.158     1.677    0.7053   6228
1+delta                1     1.906     1.266      1.25    0.1416   6223  (5 NaN)
kappa_a           0.9201     2.505     1.563     1.593    0.2151   6228
eps               0.1548    0.7831    0.3225    0.3147   0.08283   6228
Meff                   1     2.988     1.908         2    0.2601   6228


## Engineering scaling (WLS in log space)
Power law $\tau_{E,th} = \alpha_0\, I_p^{\alpha_I} B_t^{\alpha_B} \bar n_e^{\alpha_n} P_{l,th}^{\alpha_P} R_{geo}^{\alpha_R} \kappa_a^{\alpha_\kappa} \epsilon^{\alpha_\epsilon} M_{eff}^{\alpha_M}$ (paper eq. 2), fitted by weighted least squares on $\ln$-transformed data.


In [67]:
def paper_weights(tok_arr):
    """w^-1 = 2 + sqrt(n_obs,k)/4 for device k (paper section 3.2.2)."""
    w = np.ones(len(tok_arr))
    for tok in np.unique(tok_arr):
        m = tok_arr == tok
        w[m] = 1.0 / (2.0 + np.sqrt(m.sum()) / 4.0)
    return w


def wls(mask, keys):
    """WLS on ln tau_s; returns (coef, se, RMSE, R^2, MdAPE%) on the log scale."""
    y = np.log(tau_s[mask])
    X = np.column_stack([np.ones_like(y)] + [LN[k][mask] for k in keys])
    sw = np.sqrt(paper_weights(TOK[mask]))
    Xw, yw = X * sw[:, None], y * sw
    coef, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
    resid = y - X @ coef
    resid_w = yw - Xw @ coef
    n, p = X.shape
    sigma2 = np.sum(resid_w**2) / max(n - p, 1)
    cov = sigma2 * np.linalg.pinv(Xw.T @ Xw)
    se = np.sqrt(np.diag(cov))
    return (coef, se, np.sqrt(np.mean(resid**2)),
            1.0 - np.sum(resid**2) / np.sum((y - y.mean())**2),
            np.median(np.abs(resid / y)) * 100)


with np.errstate(invalid="ignore", divide="ignore"):
    LN = {"ln Ip":   np.log(ip_ma),
          "ln Bt":   np.log(Bt_T),
          "ln ne":   np.log(ne_19),
          "ln Plth": np.log(Ploss_MW),
          "ln Rgeo": np.log(RGEO),
          "ln 1+d":  np.log(one_delta),
          "ln kapa": np.log(kappa_a),
          "ln eps":  np.log(eps),
          "ln Meff": np.log(MEFF)}

keys = ["ln Ip", "ln Bt", "ln ne", "ln Plth", "ln Rgeo", "ln kapa", "ln eps", "ln Meff"]
coef, se, rmse, r2, mdape = wls(sel, keys)

# Paper Table 9, "ELMy H" row (WLS, STD5 ELMy H, no ITER-like constraint, no delta)
paper_a0 = (0.0576, 0.0013)
paper_wls = {"ln Ip": (0.984, 0.015), "ln Bt": (0.203, 0.013), "ln ne": (0.2530, 0.0091),
             "ln Plth": (-0.6658, 0.0064), "ln Rgeo": (1.698, 0.023), "ln kapa": (0.930, 0.024),
             "ln eps": (0.387, 0.032), "ln Meff": (0.175, 0.016)}

print(f"{'param':10s} {'this WLS':>20s} {'paper WLS (Tbl9)':>20s}")
a0, a0_se = np.exp(coef[0]), np.exp(coef[0]) * se[0]
print(f"{'alpha0':10s} {a0:>10.4g} pm {a0_se:<7.2g} {paper_a0[0]:>10.4g} pm {paper_a0[1]:<7.2g}")
for nm, c, s in zip(keys, coef[1:], se[1:]):
    p, ps = paper_wls[nm]
    print(f"{nm:10s} {c:>+7.3f} pm {s:<7.3f} {p:>+7.3f} pm {ps:<7.3f}")

print(f"\nN = {sel.sum()}   RMSE(log) = {rmse:.3f}   R^2 = {r2:.3f}   (paper N = 6233, RMSE = 0.18, R^2 = 0.96)")

param                  this WLS     paper WLS (Tbl9)
alpha0         0.0593 pm 0.0014      0.0576 pm 0.0013 
ln Ip       +0.973 pm 0.015    +0.984 pm 0.015  
ln Bt       +0.210 pm 0.013    +0.203 pm 0.013  
ln ne       +0.261 pm 0.009    +0.253 pm 0.009  
ln Plth     -0.667 pm 0.007    -0.666 pm 0.006  
ln Rgeo     +1.738 pm 0.024    +1.698 pm 0.023  
ln kapa     +0.911 pm 0.024    +0.930 pm 0.024  
ln eps      +0.436 pm 0.032    +0.387 pm 0.032  
ln Meff     +0.166 pm 0.017    +0.175 pm 0.016  

N = 6228   RMSE(log) = 0.188   R^2 = 0.954   (paper N = 6233, RMSE = 0.18, R^2 = 0.96)


## ITPA20-IL — ELMy H, STD5 ITER-like, with triangularity
Headline engineering scaling of the paper (eq. 5, Table 10 WLS row), extrapolated to the ITER baseline inductive $Q=10$ point.


In [57]:
il_keys = ["ln Ip", "ln Bt", "ln ne", "ln Plth", "ln Rgeo", "ln 1+d", "ln kapa", "ln eps", "ln Meff"]
coef_il, se_il, rmse_il, r2_il, mdape_il = wls(iter_like, il_keys)

itpa20_il = {"ln Ip": 1.291, "ln Bt": -0.134, "ln ne": 0.1473, "ln Plth": -0.6442, "ln Rgeo": 1.194,
             "ln 1+d": 0.560, "ln kapa": 0.673, "ln eps": -0.000, "ln Meff": 0.302}

print(f"{'param':10s} {'this WLS':>10s} {'ITPA20-IL':>11s}")
print(f"{'alpha0':10s} {np.exp(coef_il[0]):>10.4f} {0.0670:>11.4f}")
for nm, c in zip(il_keys, coef_il[1:]):
    print(f"{nm:10s} {c:>10.3f} {itpa20_il[nm]:>11.3f}")

print(f"\nN = {iter_like.sum()}   MdAPE = {mdape_il:.1f}%   RMSE(log) = {rmse_il:.3f}   R^2 = {r2_il:.3f}"
      f"   (paper N = 5260, MdAPE = 6.0%, RMSE = 0.17, R^2 = 0.95)")

# ITER baseline Q=10: Ip 15 MA, Bt 5.3 T, ne 10.3e19 m^-3, Pl,th 87 MW, Rgeo 6.2 m, delta 0.48, kappa_a 1.7, eps 0.32, Meff 2.5
iter_point = [15.0, 5.3, 10.3, 87.0, 6.2, 1.48, 1.7, 0.32, 2.5]
tau_iter = np.exp(coef_il[0] + sum(c * np.log(v) for c, v in zip(coef_il[1:], iter_point)))
print(f"\ntau_E,th(ITER) = {tau_iter:.3f} s   (paper 2.902 s)")

param        this WLS   ITPA20-IL
alpha0         0.0624      0.0670
ln Ip           1.311       1.291
ln Bt          -0.140      -0.134
ln ne           0.152       0.147
ln Plth        -0.652      -0.644
ln Rgeo         1.216       1.194
ln 1+d          0.673       0.560
ln kapa         0.620       0.673
ln eps         -0.045      -0.000
ln Meff         0.299       0.302

N = 5249   MdAPE = 6.1%   RMSE(log) = 0.169   R^2 = 0.948   (paper N = 5260, MdAPE = 6.0%, RMSE = 0.17, R^2 = 0.95)

tau_E,th(ITER) = 3.072 s   (paper 2.902 s)


## Table 8 — per-machine OLS scalings (all H-modes in STD5)
OLS on log-transformed data. 

In [58]:
from numpy.linalg import lstsq

def ols_fit(y_log, X):
    """OLS in log space; returns (coef, std_err, resid)."""
    coef, _, _, _ = lstsq(X, y_log, rcond=None)
    n, p = X.shape
    resid = y_log - X @ coef
    # unbiased variance of residuals
    sigma2 = np.sum(resid**2) / max(n - p, 1)
    cov = sigma2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    return coef, se, resid

def metrics(resid, y_log, p_free):
    """MdAPE (%), RMSE (log), R^2 — all on log scale"""
    n = len(resid)
    mdape = np.median(np.abs(resid / y_log)) * 100
    rmse  = np.sqrt(np.sum(resid**2) / n)              # no df correction, matches paper
    ss_res = np.sum(resid**2)
    ss_tot = np.sum((y_log - y_log.mean())**2)
    r2    = 1.0 - ss_res / ss_tot
    return mdape, rmse, r2

# Paper Table 8 uses "all H-modes" in STD5 (not just ELMy).
all_h = (SELDB5 == 1) & ~helium
pos_all = (tau_s > 0) & (Ploss_MW > 0) & (ip_ma > 0) & (Bt_T > 0) & (ne_19 > 0)
mask = all_h & pos_all & np.isfinite(tau_s)

# log-arrays (NaN-safe; we apply device mask before building X)
LOG = {
    "Ip":    np.log(np.where(ip_ma    > 0, ip_ma,    np.nan)),
    "Bt":    np.log(np.where(Bt_T     > 0, Bt_T,     np.nan)),
    "ne":    np.log(np.where(ne_19    > 0, ne_19,    np.nan)),
    "Plth":  np.log(np.where(Ploss_MW > 0, Ploss_MW, np.nan)),
    "Rgeo":  np.log(np.where(RGEO     > 0, RGEO,     np.nan)),
    "kappa": np.log(np.where(kappa_a  > 0, kappa_a,  np.nan)),
    "eps":   np.log(np.where(eps      > 0, eps,       np.nan)),
    "Meff":  np.log(np.where(MEFF     > 0, MEFF,     np.nan)),
    "1+d":   np.log(np.where(one_delta > 0, one_delta, np.nan)),
}
LOG_TAU = np.log(np.where(tau_s > 0, tau_s, np.nan))

# Wall-material splits (paper Table 1): AUG-W by limiter material index (W=2),
# JET-ILW by divertor material string ("W").
DIVMAT_str = DIVMAT.astype(str)
is_aug_w   = (TOK == "AUG") & (LIMMAT == 2)
is_jet_ilw = (TOK == "JET") & (DIVMAT_str == "W")

# (label, row_mask, [regressors]) -- order and predictor sets match paper Table 8
DEVICE_SPECS = [
    ("ASDEX",           TOK == "ASDEX",               ["Ip", "Bt", "ne", "Plth", "Meff"]),
    ("ASDEX Upgrade",   (TOK == "AUG") & ~is_aug_w,   ["Ip", "Bt", "ne", "Plth"]),
    ("ASDEX Upgrade W", is_aug_w,                     ["Ip", "Bt", "ne", "Plth"]),
    ("Alcator C-Mod",   TOK == "CMOD",                ["Ip",       "ne", "Plth"]),
    ("DIII-D",          TOK == "D3D",                 ["Ip", "Bt", "ne", "Plth", "1+d", "Meff"]),
    ("JET-C",           (TOK == "JET") & ~is_jet_ilw, ["Ip", "Bt", "ne", "Plth", "kappa", "Meff"]),
    ("JET-ILW",         is_jet_ilw,                   ["Ip", "Bt", "ne", "Plth", "Meff"]),
    ("JFT-2M",          TOK == "JFT2M",               ["Ip",       "ne", "Plth", "Meff"]),
    ("JT-60U",          TOK == "JT60U",               ["Ip", "Bt", "ne", "Plth"]),
    ("MAST",            TOK == "MAST",                ["Ip",       "ne", "Plth"]),
    ("NSTX",            TOK == "NSTX",                ["Ip", "Bt", "ne", "Plth", "kappa"]),
    ("PBX-M",           TOK == "PBXM",                ["Ip",       "ne", "Plth"]),
    ("PDX",             TOK == "PDX",                 ["Ip", "Bt", "ne", "Plth"]),
]

# paper Table 8 exponent columns (in order of predictors listed above)
COL_ORDER = ["Ip", "Bt", "ne", "Plth", "1+d", "kappa", "Meff"]
COL_LABELS = [r"$\alpha_I$", r"$\alpha_B$", r"$\alpha_n$", r"$\alpha_P$", 
              r"$\alpha_{(1+\delta)}$", r"$\kappa$", r"$\alpha_M$"]

header = (f"{'Device':<16s}  {'n_obs':>5s}  "
       + "   ".join(f"{c:>9s}" for c in COL_LABELS)
       + f"  {'MdAPE%':>7s}  {'RMSE':>6s}  {'R^2':>5s}")
print(header)
print("-" * len(header))

results = {}
for label, dev_mask, pkeys in DEVICE_SPECS:
    # require all regressors for this device to be finite
    log_preds = [LOG[k] for k in pkeys]
    finite_dev = np.all([np.isfinite(lp) for lp in log_preds], axis=0)
    finite_dev &= np.isfinite(LOG_TAU)
    m = mask & dev_mask & finite_dev
    n = int(m.sum())
    if n < len(pkeys) + 2:
        print(f"{label:<16s} {n:>5d}  (too few points)")
        continue

    y = LOG_TAU[m]
    cols_X = [np.ones(n)] + [LOG[k][m] for k in pkeys]
    X = np.column_stack(cols_X)
    coef, se, resid = ols_fit(y, X)
    mdape, rmse, r2 = metrics(resid, y, len(pkeys))

    # build coefficient dict keyed by predictor name
    coef_dict = {k: (coef[i+1], se[i+1]) for i, k in enumerate(pkeys)}

    # format row: show value pm standard error (se) for each column in COL_ORDER, blank if not fitted
    row = f"{label:<16s} {n:>5d}  "
    for col in COL_ORDER:
        if col in coef_dict:
            v, s = coef_dict[col]
            row += f"  {v:+5.3f}pm{s:5.3f}"
        else:
            row += f"  {'-------------':>9s}"
    row += f"  {mdape:7.1f}  {rmse:6.2f}  {r2:5.2f}"
    print(row)
    results[label] = {"n": n, "coef": coef_dict, "mdape": mdape, "rmse": rmse, "r2": r2}


Device            n_obs  $\alpha_I$   $\alpha_B$   $\alpha_n$   $\alpha_P$   $\alpha_{(1+\delta)}$    $\kappa$   $\alpha_M$   MdAPE%    RMSE    R^2
---------------------------------------------------------------------------------------------------------------------------------------------------
ASDEX              575    +0.692pm0.057  +0.130pm0.079  +0.678pm0.043  -0.648pm0.025  -------------  -------------  +0.781pm0.061      2.9    0.17   0.71
ASDEX Upgrade     1385    +1.521pm0.039  -0.272pm0.041  +0.015pm0.020  -0.691pm0.015  -------------  -------------  -------------      5.2    0.19   0.71
ASDEX Upgrade W    760    +1.631pm0.038  -0.360pm0.047  +0.027pm0.031  -0.545pm0.014  -------------  -------------  -------------      2.8    0.12   0.89
Alcator C-Mod       82    +1.149pm0.080  -------------  +0.101pm0.091  -0.597pm0.074  -------------  -------------  -------------      2.7    0.10   0.79
DIII-D             502    +1.101pm0.047  +0.100pm0.058  +0.094pm0.035  -0.633pm0.021  +0

Table 8 is reproduced essentially exactly (all exponents within ~0.005) for ASDEX, Alcator C-Mod, JFT-2M, MAST, NSTX, PBX-M and PDX. JET-C's exponents also match closely (all differences $\leq 0.006$); 
only its `n_obs` is off, by 2, see the AUG-W/JET-C helium markdown above.

Remaining issues:

- **ASDEX Upgrade / ASDEX Upgrade W**: exponents differ by up to ~0.07 (e.g. $\alpha_I$ = 1.521/1.631 here vs 1.464/1.558 in the paper). AUG-W's `n_obs` is also short by 7, see the already mentioned note.
- **DIII-D**: exponents differ by up to 0.15 (the triangularity term, $\alpha_{1+\delta}$ = 0.546 here vs 0.696 in the paper).
- **JET-ILW** and **JT-60U**: smaller residual differences, up to ~0.04 and ~0.07 respectively.

It is unlikely these are migration or crosswalk issues. Refitting directly on the raw `h-mode_data_V5.csv`, reproduces the exact same numbers we get here for every device checked (AUG, AUG-W, DIII-D).
